In [ ]:
# ============================================================
# STAGE 4 — POST-PROCESSING & VALIDATION
# D5 — Branch B — Structural Conversion
# ============================================================
#
# Validation basis:
# - Fixed Stage 1 document-grounded reference dataset
# - Branch B parsed extraction
# - Branch B technical diagnostics
# - D5 comparison rules frozen from Validation A
# ============================================================

from google.colab import files
from pathlib import Path
from difflib import SequenceMatcher

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D5"
BRANCH_ID = "B"
BRANCH_NAME = "Structural Conversion"

EXPECTED_RECORD_COUNT = 44

EXPECTED_CATEGORY_COUNTS = {
    "Main policy measure": 4,
    "Country profile": 6,
    "Narrative quantitative observation": 16,
    "Statistical table observation": 18
}

EXPECTED_FIELDS = [
    "Category",
    "Indicator or Policy Area",
    "Value",
    "Unit",
    "Qualifier",
    "Reference Period",
    "Description",
    "Source Location"
]

REFERENCE_FIELDS = EXPECTED_FIELDS.copy()

# Frozen from D5 Validation A.
#
# Alignment:
#   - block by Category + Source Location
#   - strict identity by Indicator or Policy Area + Reference Period
#   - controlled descriptive fallback when required
#
# Value and Unit never influence alignment.

ALIGNMENT_BLOCK_FIELDS = [
    "Category",
    "Source Location"
]

STRICT_IDENTITY_FIELDS = [
    "Indicator or Policy Area",
    "Reference Period"
]

# Description remains diagnostic rather than part of
# formal record-level correctness.
PRIMARY_CORRECTNESS_FIELDS = [
    "Category",
    "Indicator or Policy Area",
    "Value",
    "Unit",
    "Qualifier",
    "Reference Period",
    "Source Location"
]

INDICATOR_MATCH_THRESHOLD = 0.50
DESCRIPTION_MATCH_THRESHOLD = 0.35
FALLBACK_TOTAL_THRESHOLD = 0.45

NUMERIC_TOLERANCE = 1e-9

OUTPUT_DIR = Path(
    "outputs_D5_validation_branch_B"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Validation configured.")
print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH_ID, "-", BRANCH_NAME)

In [ ]:
# ============================================================
# 2. Upload validation inputs
# ============================================================
# Required:
#   1) D5_reference_values.csv
#   2) D5_branch_B_parsed_extraction.json
#   3) D5_branch_B_technical_diagnostics.json

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [
    f for f in uploaded_files
    if f.lower().endswith(".csv")
]

json_files = [
    f for f in uploaded_files
    if f.lower().endswith(".json")
]

if len(csv_files) != 1:
    raise ValueError(
        "Upload exactly one D5 Stage 1 reference CSV."
    )

if len(json_files) != 2:
    raise ValueError(
        "Upload exactly two JSON files: the Branch B parsed "
        "extraction and technical diagnostics."
    )

REFERENCE_FILE = csv_files[0]

PARSED_EXTRACTION_FILE = None
TECHNICAL_DIAGNOSTICS_FILE = None

for file_name in json_files:

    with open(
        file_name,
        "r",
        encoding="utf-8-sig"
    ) as f:
        obj = json.load(f)

    if not isinstance(obj, dict):
        continue

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and isinstance(
            obj.get("records"),
            list
        )
    ):
        PARSED_EXTRACTION_FILE = file_name

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and "structurally_evaluable" in obj
        and "record_schema_valid" in obj
        and "valid_json" in obj
    ):
        TECHNICAL_DIAGNOSTICS_FILE = file_name


if PARSED_EXTRACTION_FILE is None:
    raise ValueError(
        "Could not identify the D5 Branch B parsed extraction."
    )

if TECHNICAL_DIAGNOSTICS_FILE is None:
    raise ValueError(
        "Could not identify the D5 Branch B technical diagnostics."
    )

print("Reference:", REFERENCE_FILE)
print(
    "Parsed extraction:",
    PARSED_EXTRACTION_FILE
)
print(
    "Technical diagnostics:",
    TECHNICAL_DIAGNOSTICS_FILE
)

In [ ]:
# ============================================================
# 3. Load inputs and verify identity/provenance
# ============================================================

with open(
    PARSED_EXTRACTION_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    extraction_json = json.load(f)

with open(
    TECHNICAL_DIAGNOSTICS_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    technical_diagnostics = json.load(f)


reference_df = pd.read_csv(
    REFERENCE_FILE,
    dtype=object,
    keep_default_na=False,
    encoding="utf-8-sig"
)


def restore_null(value):
    return (
        None
        if value == ""
        else value
    )


for column in reference_df.columns:
    reference_df[column] = (
        reference_df[column]
        .map(restore_null)
    )


extracted_df = pd.DataFrame(
    extraction_json["records"]
)


for artefact_name, artefact in {
    "parsed extraction":
        extraction_json,

    "technical diagnostics":
        technical_diagnostics
}.items():

    if artefact.get(
        "document_id"
    ) != DOCUMENT_ID:
        raise ValueError(
            f"Unexpected {artefact_name} document_id: "
            f"{artefact.get('document_id')}"
        )

    if artefact.get(
        "branch"
    ) != BRANCH_ID:
        raise ValueError(
            f"Unexpected {artefact_name} branch: "
            f"{artefact.get('branch')}"
        )


def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(8192),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


input_provenance = {
    "reference_file":
        REFERENCE_FILE,

    "reference_sha256":
        sha256_file(
            REFERENCE_FILE
        ),

    "parsed_extraction_file":
        PARSED_EXTRACTION_FILE,

    "parsed_extraction_sha256":
        sha256_file(
            PARSED_EXTRACTION_FILE
        ),

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_FILE,

    "technical_diagnostics_sha256":
        sha256_file(
            TECHNICAL_DIAGNOSTICS_FILE
        )
}

print(
    "Reference shape:",
    reference_df.shape
)

print(
    "Extraction shape:",
    extracted_df.shape
)

In [ ]:
# ============================================================
# 4. Import Branch B technical/schema status
# ============================================================

structurally_evaluable = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

schema_validity = bool(
    structurally_evaluable
)

schema_diagnostics = {
    "valid_json":
        bool(
            technical_diagnostics.get(
                "valid_json",
                False
            )
        ),

    "record_schema_valid":
        bool(
            technical_diagnostics.get(
                "record_schema_valid",
                False
            )
        ),

    "field_types_valid":
        bool(
            technical_diagnostics.get(
                "field_types_valid",
                False
            )
        ),

    "structurally_evaluable":
        structurally_evaluable,

    "schema_validity":
        schema_validity
}


if not structurally_evaluable:
    raise ValueError(
        "D5 Branch B output is not structurally evaluable. "
        "Content-level validation cannot proceed."
    )


print(
    "Imported Branch B technical/schema status:"
)

print(
    json.dumps(
        schema_diagnostics,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 5. Verify fixed Stage 1 reference and extraction fields
# ============================================================

if (
    reference_df.columns.tolist()
    != REFERENCE_FIELDS
):
    raise ValueError(
        "D5 Stage 1 reference schema "
        "does not match expected fields."
    )


if (
    len(reference_df)
    != EXPECTED_RECORD_COUNT
):
    raise ValueError(
        f"Expected {EXPECTED_RECORD_COUNT} Stage 1 records, "
        f"found {len(reference_df)}."
    )


reference_category_counts = (
    reference_df[
        "Category"
    ]
    .value_counts()
    .to_dict()
)


if (
    reference_category_counts
    != EXPECTED_CATEGORY_COUNTS
):
    raise ValueError(
        "D5 Stage 1 category counts "
        "do not match the fixed design."
    )


missing_extraction_fields = [
    field
    for field in EXPECTED_FIELDS
    if field not in extracted_df.columns
]


extraction_cmp = (
    extracted_df.copy(
        deep=True
    )
)


# Missing fields are introduced only
# in the comparison copy.
for field in missing_extraction_fields:
    extraction_cmp[field] = None


extraction_cmp = (
    extraction_cmp[
        EXPECTED_FIELDS
    ].copy()
)


reference_cmp = (
    reference_df[
        REFERENCE_FIELDS
    ].copy()
)


print(
    "Reference records:",
    len(reference_cmp)
)

print(
    "Extracted records:",
    len(extraction_cmp)
)

print(
    "Missing extraction columns:",
    missing_extraction_fields
)

In [ ]:
# ------------------------------------------------------------
# 6. Controlled comparison normalisation
# ------------------------------------------------------------

DASH_REPLACEMENTS = {
    "\u2010": "-",
    "\u2011": "-",
    "\u2012": "-",
    "\u2013": "-",
    "\u2014": "-",
    "\u2212": "-"
}

APOSTROPHE_REPLACEMENTS = {
    "\u2018": "'",
    "\u2019": "'",
    "\u02bc": "'",
    "`": "'"
}

def normalise_text(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None

    text = unicodedata.normalize("NFKC", str(value))
    text = (
        text
        .replace("\u00a0", " ")
        .replace("\u2007", " ")
        .replace("\u202f", " ")
    )

    for source, target in DASH_REPLACEMENTS.items():
        text = text.replace(source, target)

    for source, target in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(source, target)

    text = re.sub(r"[ \t\f\v]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)

    return text.strip().casefold()

INDICATOR_EQUIVALENCE_RAW = {
    "Price and financial stability": [
        "Managing price and financial stability",
    ],
    "Fiscal sustainability while financing development": [
        "Achieving fiscal sustainability while financing development",
    ],
    "Financial sector development": [
        "Broadening and deepening the financial sector",
    ],
    "Job-intensive inclusive growth": [
        "Promoting more job-intensive, inclusive growth",
    ],
    "Recent wholesale price inflation": [
        "Wholesale price index annual rise",
    ],
    "Wholesale price inflation one year earlier": [
        "Wholesale price index annual rise",
    ],
    "Previous high general government deficit": [
        "General government deficit high",
    ],
    "Agricultural labour-force share": [
        "Labor force working in agriculture",
    ],
    "Population living on less than USD 2 a day": [
        "Indians living on less than $2 a day",
    ],
}


INDICATOR_EQUIVALENCE = {
    normalise_text(reference): {
        normalise_text(variant)
        for variant in variants
    }
    for reference, variants in INDICATOR_EQUIVALENCE_RAW.items()
}


def indicator_equivalent(reference_value, extracted_value):
    ref = normalise_text(reference_value)
    ext = normalise_text(extracted_value)

    # Normalised exact match
    if ref == ext:
        return True

    # Predefined source-grounded equivalent label
    return ext in INDICATOR_EQUIVALENCE.get(ref, set())

STOPWORDS = {
    "a","an","and","as","at","by","for","from","in","into","is","of",
    "on","or","the","to","was","were","with","reported","reporting",
    "described","principal","measure","observation","india","indias"
}

def comparison_tokens(value):
    text = normalise_text(value)
    if text is None:
        return set()

    text = re.sub(r"[^a-z0-9]+", " ", text)
    return {
        token for token in text.split()
        if token and token not in STOPWORDS
    }

def text_similarity(first_value, second_value):
    first_text = normalise_text(first_value)
    second_text = normalise_text(second_value)

    if first_text is None and second_text is None:
        return 1.0
    if first_text is None or second_text is None:
        return 0.0
    if first_text == second_text:
        return 1.0

    first_tokens = comparison_tokens(first_value)
    second_tokens = comparison_tokens(second_value)

    if first_tokens or second_tokens:
        token_score = (
            len(first_tokens & second_tokens) / len(first_tokens | second_tokens)
        )
    else:
        token_score = 0.0

    sequence_score = SequenceMatcher(
        None, first_text, second_text
    ).ratio()

    return max(token_score, sequence_score)


def numeric_value(value):
    if value is None or isinstance(value, bool):
        return None

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = str(value).strip()
    text = (
        text
        .replace(",", "")
        .replace("$", "")
        .replace("–", "-")
        .replace("−", "-")
    )

    if re.fullmatch(r"-?\d+(?:\.\d+)?", text):
        return float(text)

    return None


def values_match(reference_value, extracted_value):
    ref_num = numeric_value(reference_value)
    ext_num = numeric_value(extracted_value)

    if ref_num is not None and ext_num is not None:
        return math.isclose(
            ref_num, ext_num,
            rel_tol=0.0,
            abs_tol=NUMERIC_TOLERANCE
        )

    return normalise_text(reference_value) == normalise_text(extracted_value)


In [ ]:
# ------------------------------------------------------------
# 7. Controlled unit, qualifier and period comparison
# ------------------------------------------------------------

UNIT_EQUIVALENCE_MAP = {
    "usd": "usd",
    "dollars": "usd",
    "dollar": "usd",
    "billion": "billion people",
    "billion people": "billion people",
    "million people": "million people",
    "million indians": "million people",
    "usd billion": "billion dollars",
    "billion dollars": "billion dollars",
    "percent of labor force": "percent of labour force",
    "percent of the labor force": "percent of labour force",
    "percent of labour force": "percent of labour force",
    "percent of the labour force": "percent of labour force"
}

def normalise_unit(value):
    norm = normalise_text(value)
    if norm is None:
        return None
    return UNIT_EQUIVALENCE_MAP.get(norm, norm)

def units_match(reference_value, extracted_value):
    return normalise_unit(reference_value) == normalise_unit(extracted_value)


def normalise_qualifier(value):
    norm = normalise_text(value)
    if norm is None:
        return None
    return norm.rstrip(".")

def qualifiers_match(reference_value, extracted_value):
    return normalise_qualifier(reference_value) == normalise_qualifier(extracted_value)


FISCAL_PERIOD_PATTERN = re.compile(
    r"\b((?:19|20)\d{2})\s*[/–-]\s*(\d{2})\b"
)

def canonical_relative_period(value):
    text = normalise_text(value)

    if text is None:
        return None

    match = FISCAL_PERIOD_PATTERN.search(str(value))
    if match:
        return f"{match.group(1)}/{match.group(2)}"

    equivalence = {
        "four years running": "four years running",
        "next 10 years": "next 10 years",
        "over the next 10 years": "next 10 years",
        "since 2002": "since 2002",
        "recently": "recently",
        "a year ago": "a year ago",
        "one year earlier": "a year ago",
        "medium term": "medium term",
        "over the medium term": "medium term",
        "after the latest tightening": "current tightening",
        "just last week": "current tightening",
        "now": "current tightening",
        "before the latest tightening": "previous tightening",
        "previously": "previous tightening",
        "earlier": "previous tightening"
    }

    return equivalence.get(text, text)

def periods_match(reference_value, extracted_value):
    return (
        canonical_relative_period(reference_value)
        == canonical_relative_period(extracted_value)
    )


In [ ]:
# ------------------------------------------------------------
# 8. Prepare alignment blocks
# ------------------------------------------------------------

for df in (
    reference_cmp,
    extraction_cmp
):

    df["_category_norm"] = (
        df["Category"]
        .map(normalise_text)
    )

    df["_source_norm"] = (
        df["Source Location"]
        .map(normalise_text)
    )

    df["_indicator_norm"] = (
        df[
            "Indicator or Policy Area"
        ]
        .map(normalise_text)
    )

    df["_period_norm"] = (
        df[
            "Reference Period"
        ]
        .map(
            canonical_relative_period
        )
    )

    df["_block"] = list(
        zip(
            df["_category_norm"],
            df["_source_norm"]
        )
    )


reference_cmp[
    "_reference_index"
] = range(
    len(reference_cmp)
)


extraction_cmp[
    "_extraction_index"
] = range(
    len(extraction_cmp)
)


print("Identity blocks prepared.")


In [ ]:
# ------------------------------------------------------------
# 9. Deterministic one-to-one alignment
# ------------------------------------------------------------

matched_pairs = []
used_ref = set()
used_ext = set()

all_blocks = sorted(
    set(reference_cmp["_block"]) | set(extraction_cmp["_block"]),
    key=str
)

for block in all_blocks:
    ref_block = reference_cmp[reference_cmp["_block"] == block]
    ext_block = extraction_cmp[extraction_cmp["_block"] == block]

    if ref_block.empty or ext_block.empty:
        continue

    # Exact descriptive identity first
    ext_lookup = {}
    for _, ext_row in ext_block.iterrows():
        key = (
            ext_row["_indicator_norm"],
            ext_row["_period_norm"]
        )
        ext_lookup.setdefault(key, []).append(int(ext_row["_extraction_index"]))

    for _, ref_row in ref_block.iterrows():
        ref_idx = int(ref_row["_reference_index"])
        key = (
            ref_row["_indicator_norm"],
            ref_row["_period_norm"]
        )

        candidates = [
            idx for idx in ext_lookup.get(key, [])
            if idx not in used_ext
        ]

        if candidates:
            ext_idx = candidates[0]
            used_ref.add(ref_idx)
            used_ext.add(ext_idx)
            matched_pairs.append({
                "reference_index": ref_idx,
                "extraction_index": ext_idx,
                "match_method": "strict_identity",
                "alignment_score": 1.0
            })

    remaining_refs = [
        row for _, row in ref_block.iterrows()
        if int(row["_reference_index"]) not in used_ref
    ]
    remaining_exts = [
        row for _, row in ext_block.iterrows()
        if int(row["_extraction_index"]) not in used_ext
    ]

    if not remaining_refs or not remaining_exts:
        continue

    cost_matrix = []
    details_matrix = []

    for ref_row in remaining_refs:
        cost_row = []
        details_row = []

        for ext_row in remaining_exts:
            indicator_score = text_similarity(
                ref_row["Indicator or Policy Area"],
                ext_row["Indicator or Policy Area"]
            )

            description_score = text_similarity(
                ref_row["Description"],
                ext_row["Description"]
            )

            period_score = (
                1.0
                if periods_match(
                    ref_row["Reference Period"],
                    ext_row["Reference Period"]
                )
                else 0.0
            )

            total = (
                0.70 * indicator_score
                + 0.25 * description_score
                + 0.05 * period_score
            )

            cost_row.append(1.0 - total)
            details_row.append({
                "total": total,
                "indicator": indicator_score,
                "description": description_score,
                "period": period_score
            })

        cost_matrix.append(cost_row)
        details_matrix.append(details_row)

    row_idx, col_idx = linear_sum_assignment(cost_matrix)

    for rpos, cpos in zip(row_idx, col_idx):
        ref_row = remaining_refs[rpos]
        ext_row = remaining_exts[cpos]
        details = details_matrix[rpos][cpos]

        if details["total"] < FALLBACK_TOTAL_THRESHOLD:
            continue

        if (
            details["indicator"] < INDICATOR_MATCH_THRESHOLD
            and details["description"] < DESCRIPTION_MATCH_THRESHOLD
        ):
            continue

        ref_idx = int(ref_row["_reference_index"])
        ext_idx = int(ext_row["_extraction_index"])

        if ref_idx in used_ref or ext_idx in used_ext:
            continue

        used_ref.add(ref_idx)
        used_ext.add(ext_idx)

        matched_pairs.append({
            "reference_index": ref_idx,
            "extraction_index": ext_idx,
            "match_method": "descriptive_fallback",
            "alignment_score": details["total"],
            "indicator_alignment_score": details["indicator"],
            "description_alignment_score": details["description"],
            "period_disambiguation_score": details["period"]
        })

matched_pairs = sorted(matched_pairs, key=lambda x: x["reference_index"])

missing_reference_indices = [
    idx for idx in reference_cmp["_reference_index"].tolist()
    if idx not in used_ref
]

unsupported_extraction_indices = [
    idx for idx in extraction_cmp["_extraction_index"].tolist()
    if idx not in used_ext
]

print("Aligned records:", len(matched_pairs))
print("Missing reference records:", len(missing_reference_indices))
print("Unsupported extracted records:", len(unsupported_extraction_indices))
print(pd.Series(
    [pair["match_method"] for pair in matched_pairs],
    dtype="object"
).value_counts())


In [ ]:
# ============================================================
# 10. Field-level correctness after alignment
# ============================================================

comparison_rows = []
field_rows = []


for pair in matched_pairs:

    ref = reference_cmp.loc[
        reference_cmp[
            "_reference_index"
        ]
        == pair[
            "reference_index"
        ]
    ].iloc[0]

    ext = extraction_cmp.loc[
        extraction_cmp[
            "_extraction_index"
        ]
        == pair[
            "extraction_index"
        ]
    ].iloc[0]


    field_matches = {
        "Category":
            (
                normalise_text(
                    ref["Category"]
                )
                ==
                normalise_text(
                    ext["Category"]
                )
            ),

        "Indicator or Policy Area":
            indicator_equivalent(
                ref[
                    "Indicator or Policy Area"
                ],
                ext[
                    "Indicator or Policy Area"
                ]
            ),

        "Value":
            values_match(
                ref["Value"],
                ext["Value"]
            ),

        "Unit":
            units_match(
                ref["Unit"],
                ext["Unit"]
            ),

        "Qualifier":
            qualifiers_match(
                ref["Qualifier"],
                ext["Qualifier"]
            ),

        "Reference Period":
            periods_match(
                ref[
                    "Reference Period"
                ],
                ext[
                    "Reference Period"
                ]
            ),

        "Description":
            (
                text_similarity(
                    ref["Description"],
                    ext["Description"]
                )
                >= 0.70
            ),

        "Source Location":
            (
                normalise_text(
                    ref["Source Location"]
                )
                ==
                normalise_text(
                    ext["Source Location"]
                )
            )
    }


    all_primary_fields_match = all(
        field_matches[field]
        for field
        in PRIMARY_CORRECTNESS_FIELDS
    )


    comparison_rows.append({
        "Reference Record ID":
            (
                f"D5-REF-"
                f"{pair['reference_index'] + 1:03d}"
            ),

        "Extraction Record ID":
            (
                f"D5-B-"
                f"{pair['extraction_index'] + 1:03d}"
            ),

        "match_method":
            pair[
                "match_method"
            ],

        "alignment_score":
            pair.get(
                "alignment_score"
            ),

        **{
            f"{field}_ref":
                ref[field]
            for field
            in EXPECTED_FIELDS
        },

        **{
            f"{field}_ext":
                ext[field]
            for field
            in EXPECTED_FIELDS
        },

        **{
            f"{field}_match":
                match
            for field, match
            in field_matches.items()
        },

        "all_primary_fields_match":
            all_primary_fields_match,

        "record_status":
            (
                "fully_correct"
                if all_primary_fields_match
                else "discrepant"
            ),

        "all_mismatched_fields":
            ", ".join(
                field
                for field, match
                in field_matches.items()
                if not match
            ),

        "primary_mismatched_fields":
            ", ".join(
                field
                for field
                in PRIMARY_CORRECTNESS_FIELDS
                if not field_matches[field]
            )
    })


    for field in EXPECTED_FIELDS:

        field_rows.append({
            "Reference Record ID":
                (
                    f"D5-REF-"
                    f"{pair['reference_index'] + 1:03d}"
                ),

            "Extraction Record ID":
                (
                    f"D5-B-"
                    f"{pair['extraction_index'] + 1:03d}"
                ),

            "Field":
                field,

            "Field Match":
                field_matches[field],

            "match_method":
                pair[
                    "match_method"
                ]
        })


comparison_df = pd.DataFrame(
    comparison_rows
)

field_validation_df = pd.DataFrame(
    field_rows
)

comparison_df.head()

In [ ]:
# ============================================================
# 11. Create outcome datasets
# ============================================================

missing_records = (
    reference_cmp[
        reference_cmp[
            "_reference_index"
        ].isin(
            missing_reference_indices
        )
    ].copy()
)


unsupported_records = (
    extraction_cmp[
        extraction_cmp[
            "_extraction_index"
        ].isin(
            unsupported_extraction_indices
        )
    ].copy()
)


fully_correct_records = (
    comparison_df[
        comparison_df[
            "record_status"
        ] == "fully_correct"
    ].copy()
)


discrepant_records = (
    comparison_df[
        comparison_df[
            "record_status"
        ] == "discrepant"
    ].copy()
)


for dataframe in (
    missing_records,
    unsupported_records
):

    helper_columns = [
        column
        for column in dataframe.columns
        if column.startswith("_")
    ]

    if helper_columns:

        dataframe.drop(
            columns=helper_columns,
            inplace=True
        )


print(
    "Fully correct:",
    len(
        fully_correct_records
    )
)

print(
    "Discrepant:",
    len(
        discrepant_records
    )
)

print(
    "Missing:",
    len(
        missing_records
    )
)

print(
    "Unsupported/unmatched:",
    len(
        unsupported_records
    )
)

In [ ]:
# ============================================================
# 12. Calculate common validation metrics
# ============================================================

N_REF = int(
    len(reference_cmp)
)

N_EXT = int(
    len(extraction_cmp)
)

N_ALIGNED = int(
    len(comparison_df)
)

N_CORRECT = int(
    len(
        fully_correct_records
    )
)

N_DISCREPANT = int(
    len(
        discrepant_records
    )
)

N_MISSING = int(
    len(
        missing_records
    )
)

N_UNSUPPORTED = int(
    len(
        unsupported_records
    )
)


completeness = (
    N_ALIGNED / N_REF
    if N_REF
    else 0.0
)

missing_rate = (
    N_MISSING / N_REF
    if N_REF
    else 0.0
)


record_precision = (
    N_CORRECT / N_EXT
    if N_EXT
    else 0.0
)

record_recall = (
    N_CORRECT / N_REF
    if N_REF
    else 0.0
)

record_f1 = (
    2
    * record_precision
    * record_recall
    / (
        record_precision
        + record_recall
    )
    if (
        record_precision
        + record_recall
    )
    else 0.0
)


unsupported_rate = (
    N_UNSUPPORTED / N_EXT
    if N_EXT
    else 0.0
)

discrepancy_rate = (
    N_DISCREPANT
    / N_ALIGNED
    if N_ALIGNED
    else 0.0
)


field_accuracy_among_aligned = {}

for field in EXPECTED_FIELDS:

    rows = (
        field_validation_df[
            field_validation_df[
                "Field"
            ] == field
        ]
    )

    field_accuracy_among_aligned[
        field
    ] = (
        float(
            rows[
                "Field Match"
            ].mean()
        )
        if len(rows)
        else 0.0
    )


primary_field_validation_df = (
    field_validation_df[
        field_validation_df[
            "Field"
        ].isin(
            PRIMARY_CORRECTNESS_FIELDS
        )
    ]
)


correct_field_instances = int(
    primary_field_validation_df[
        "Field Match"
    ].sum()
)


expected_field_instances = int(
    N_REF
    * len(
        PRIMARY_CORRECTNESS_FIELDS
    )
)


field_accuracy = (
    correct_field_instances
    / expected_field_instances
    if expected_field_instances
    else 0.0
)


print(
    "Common validation metrics calculated."
)

In [ ]:
# ============================================================
# 13. Field- and category-level diagnostics
# ============================================================

field_error_summary = []


for field in EXPECTED_FIELDS:

    rows = (
        field_validation_df[
            field_validation_df[
                "Field"
            ] == field
        ]
    )

    correct = int(
        rows[
            "Field Match"
        ].sum()
    )

    incorrect = int(
        len(rows)
        - correct
    )


    field_error_summary.append({
        "field":
            field,

        "used_in_alignment_block":
            field
            in ALIGNMENT_BLOCK_FIELDS,

        "used_in_strict_identity":
            field
            in STRICT_IDENTITY_FIELDS,

        "used_in_primary_correctness":
            field
            in PRIMARY_CORRECTNESS_FIELDS,

        "aligned_records_evaluated":
            int(
                len(rows)
            ),

        "correct_values_among_aligned":
            correct,

        "incorrect_values_among_aligned":
            incorrect,

        "accuracy_among_aligned":
            (
                round(
                    correct
                    / len(rows),
                    4
                )
                if len(rows)
                else 0.0
            ),

        "missing_expected_instances":
            N_MISSING,

        "overall_expected_instances":
            N_REF,

        "overall_accuracy_against_reference":
            (
                round(
                    correct
                    / N_REF,
                    4
                )
                if N_REF
                else 0.0
            )
    })


field_error_summary_df = pd.DataFrame(
    field_error_summary
)


category_rows = []


for (
    category,
    expected_count
) in EXPECTED_CATEGORY_COUNTS.items():

    ref_indices = set(
        reference_cmp.loc[
            reference_cmp[
                "Category"
            ] == category,
            "_reference_index"
        ].tolist()
    )


    aligned_category = (
        comparison_df[
            comparison_df[
                "Reference Record ID"
            ].apply(
                lambda value:
                    (
                        int(
                            str(value)
                            .split("-")[-1]
                        )
                        - 1
                    )
                    in ref_indices
            )
        ]
    )


    category_rows.append({
        "Category":
            category,

        "Expected Records":
            expected_count,

        "Aligned Records":
            len(
                aligned_category
            ),

        "Fully Correct Records":
            int(
                (
                    aligned_category[
                        "record_status"
                    ]
                    == "fully_correct"
                ).sum()
            ),

        "Discrepant Records":
            int(
                (
                    aligned_category[
                        "record_status"
                    ]
                    == "discrepant"
                ).sum()
            ),

        "Missing Records":
            (
                expected_count
                - len(
                    aligned_category
                )
            )
    })


category_metrics_df = pd.DataFrame(
    category_rows
)

display(
    field_error_summary_df
)

display(
    category_metrics_df
)

In [ ]:
# ============================================================
# 14. Build final Branch B validation summary
# ============================================================

summary = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH_ID,

    "branch_name":
        BRANCH_NAME,

    "reference_records":
        N_REF,

    "extracted_records":
        N_EXT,

    "aligned_records":
        N_ALIGNED,

    "fully_correct_records":
        N_CORRECT,

    "discrepant_records":
        N_DISCREPANT,

    "missing_records":
        N_MISSING,

    "unsupported_extracted_records":
        N_UNSUPPORTED,

    "completeness":
        round(
            completeness,
            4
        ),

    "missing_rate":
        round(
            missing_rate,
            4
        ),

    "record_precision_exact":
        round(
            record_precision,
            4
        ),

    "record_recall_exact":
        round(
            record_recall,
            4
        ),

    "record_f1_exact":
        round(
            record_f1,
            4
        ),

    "unsupported_rate":
        round(
            unsupported_rate,
            4
        ),

    "discrepancy_rate_among_aligned":
        round(
            discrepancy_rate,
            4
        ),

    "field_accuracy":
        round(
            field_accuracy,
            4
        ),

    "description_diagnostic_accuracy":
        round(
            field_accuracy_among_aligned[
                "Description"
            ],
            4
        ),

    "field_accuracy_among_aligned": {
        key:
            round(
                value,
                4
            )
        for key, value
        in field_accuracy_among_aligned.items()
    },

    "schema_validity":
        schema_validity,

    "schema_diagnostics":
        schema_diagnostics,

    "structurally_evaluable":
        structurally_evaluable,

    "alignment_block_fields":
        ALIGNMENT_BLOCK_FIELDS,

    "strict_identity_fields":
        STRICT_IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "matching_rules": {
        "blocking_fields":
            ALIGNMENT_BLOCK_FIELDS,

        "strict_identity_fields":
            STRICT_IDENTITY_FIELDS,

        "fallback_fields": [
            "Indicator or Policy Area",
            "Description"
        ],

        "reference_period_role":
            (
                "Strict identity field and weak "
                "fallback disambiguation signal"
            ),

        "value_used_for_alignment":
            False,

        "unit_used_for_alignment":
            False,

        "indicator_match_threshold":
            INDICATOR_MATCH_THRESHOLD,

        "description_match_threshold":
            DESCRIPTION_MATCH_THRESHOLD,

        "fallback_total_threshold":
            FALLBACK_TOTAL_THRESHOLD
    },

    "comparison_rules_frozen_from_branch_A":
        True,

    "comparison_rules": {
        "value":
            (
                "Numeric equality where numeric; "
                "controlled normalised text otherwise"
            ),

        "unit":
            (
                "Controlled source-grounded "
                "equivalence map"
            ),

        "qualifier":
            (
                "Case/punctuation "
                "normalisation only"
            ),

        "reference_period":
            (
                "Controlled source-grounded "
                "canonical equivalence"
            ),

        "indicator":
            (
                "Normalised exact comparison supplemented "
                "by the predefined source-grounded D5 "
                "equivalence map frozen in Validation A"
            ),

        "indicator_equivalence_rules_frozen_across_branches":
            True,

        "description":
            (
                "Deterministic lexical similarity after "
                "alignment; similarity >= 0.70 is "
                "diagnostic only"
            ),

        "description_similarity_threshold":
            0.70,

        "source_location":
            (
                "Normalised exact comparison "
                "after alignment"
            )
    },

    "normalisation_note":
        (
            "Deterministic normalisation was applied only "
            "to comparison copies; the preserved Branch B "
            "extraction was not modified."
        ),

    "input_provenance":
        input_provenance
}


print(
    "Final validation summary:"
)

print(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 15. Validation integrity checks
# ============================================================

assert (
    N_ALIGNED
    + N_MISSING
    == N_REF
)

assert (
    N_ALIGNED
    + N_UNSUPPORTED
    == N_EXT
)

assert (
    N_CORRECT
    + N_DISCREPANT
    == N_ALIGNED
)


for metric_name, metric_value in {
    "completeness":
        completeness,

    "missing_rate":
        missing_rate,

    "record_precision":
        record_precision,

    "record_recall":
        record_recall,

    "record_f1":
        record_f1,

    "unsupported_rate":
        unsupported_rate,

    "discrepancy_rate":
        discrepancy_rate,

    "field_accuracy":
        field_accuracy
}.items():

    assert (
        0.0
        <= metric_value
        <= 1.0
    ), (
        f"Invalid {metric_name}: "
        f"{metric_value}"
    )


print(
    "Validation integrity checks passed."
)

In [ ]:
# ============================================================
# 16. Export validation artefacts
# ============================================================

comparison_df.to_csv(
    OUTPUT_DIR
    / "D5_branch_B_validation_detailed.csv",
    index=False
)


field_validation_df.to_csv(
    OUTPUT_DIR
    / "D5_branch_B_field_validation.csv",
    index=False
)


missing_records.to_csv(
    OUTPUT_DIR
    / "D5_branch_B_missing_records.csv",
    index=False
)


unsupported_records.to_csv(
    OUTPUT_DIR
    / "D5_branch_B_unsupported_records.csv",
    index=False
)


fully_correct_records.to_csv(
    OUTPUT_DIR
    / "D5_branch_B_fully_correct_records.csv",
    index=False
)


discrepant_records.to_csv(
    OUTPUT_DIR
    / "D5_branch_B_discrepant_records.csv",
    index=False
)


field_error_summary_df.to_csv(
    OUTPUT_DIR
    / "D5_branch_B_field_error_summary.csv",
    index=False
)


category_metrics_df.to_csv(
    OUTPUT_DIR
    / "D5_branch_B_category_metrics.csv",
    index=False
)


with open(
    OUTPUT_DIR
    / "D5_branch_B_validation_summary.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )


print(
    "Validation artefacts saved."
)

In [ ]:
# ------------------------------------------------------------
# 17. Download generated validation artefacts
# ------------------------------------------------------------

for output_file in sorted(OUTPUT_DIR.iterdir()):
    if output_file.is_file():
        files.download(output_file)
